# RAG pipeline - interactive driver notebook

All logic lives in `rag_core.py` (single source of truth); this notebook only calls it,
so the two can never drift apart. 

Sections: **Setup** -> **Inspect** -> **Single mode** -> **Batch mode**.

In [ ]:
# ── Setup: configure this run  (THIS is where you change the BATCH-MODE dataset) ──
# Uncomment + edit the two PATH lines to switch e.g. sub01 -> sub02.
# NOTE: if you already ran this cell, restart the kernel first (env vars are read at import).
# Set env vars BEFORE importing rag_core; identical to running rag_core.py in a terminal.
import os

# os.environ["IAM_LLM_MODEL"] = "gpt-4o-mini"          # default: gpt-4.1-mini
# os.environ["IAM_LLM_BATCH_SHEET_PATH"] = "data/split_outputs_6351/AR6_6351_sub01_test_list.xlsx"
# os.environ["IAM_LLM_GT_PATH"]          = "data/split_outputs_6351/AR6_6351_sub01_test_gt.xlsx"

import rag_core
print("model      :", rag_core.MODEL_NAME)
print("topk       :", rag_core.TOPK_SCEN, rag_core.TOPK_MODEL, rag_core.TOPK_CROSS, "| step3:", rag_core.RUN_STEP3)
print("fp sheet   :", rag_core.FP_SHEET_NAME, "| scenario desc:", rag_core.INCLUDE_SCENARIO_DESC)
print("test list  :", rag_core.BATCH_SHEET_PATH)
print("RUN_TAG    :", rag_core.RUN_TAG)

## Inspect one target

In [ ]:
# ── Pick a target  (THIS is where you change the SINGLE-MODE object) ──
# Option A: take row IDX from the current test list
IDX = 0
dfv = rag_core.load_validation_xlsx(rag_core.BATCH_SHEET_PATH)
row = dfv.iloc[IDX]
target = {"region": row["region"], "model_family": row["model_family"], "scenario": row["scenario"]}

# Option B: write it by hand (uncomment and edit; values must match the data exactly)
# target = {
#     "region": "Countries of centrally-planned Asia; primarily China",
#     "model_family": "IMAGE",
#     "scenario": "CO_BAU",
# }

target

In [ ]:
# ── Retrieved neighbors for this target ──
steps = rag_core.compute_steps_for_target(target)
print("STEP1 (same region+family, similar scenarios):", len(steps["STEP1_DF"]), "rows")
print("STEP2 (same region+scenario, other families) :", len(steps["STEP2_DF"]), "rows  <- may be 0 under the 6351 split; that is expected")
print("STEP3 (cross)                                :", len(steps["STEP3_DF"]), "rows")
steps["STEP2_DF"].head(20) if not steps["STEP2_DF"].empty else "STEP2 empty: no same-scenario neighbor for this target"

In [ ]:
# ── Full prompt preview ──
nb_raw = rag_core.concat_neighbor_raw(
    steps["STEP1_DF"], steps["STEP2_DF"],
    steps["STEP3_DF"] if not steps["STEP3_DF"].empty else None)
neighbors_all = rag_core.build_neighbors_all_evidence_hard(nb_raw, rag_core.VARS_OUT)
prompt = rag_core.build_prompt(
    target_block=rag_core.build_target_block(target["model_family"], target["scenario"]),
    neighbors_all=rag_core.round_dict_floats(neighbors_all, sig=2),
    vars_out=rag_core.VARS_OUT,
    total_var=rag_core.TOTAL_ELECTRICITY_VAR,
    tech_vars=rag_core.TECH_ELECTRICITY_VARS,
    agg_identities=rag_core.AGG_IDENTITIES,
    include_cgroup_rationale=False,
    include_per_var_rationale=False,
)
print(f"prompt length: {len(prompt)} chars\n")
print(prompt[:3000])

## Single mode

One target, WITH c-group and per-variable rationales printed.
Saves `outputs/synthetic_target_output_hardmode.json` / `.xlsx`. Costs one LLM call.

In [ ]:
# ── Single mode (equivalent to MODE='single' in rag_core.py) ──
rag_core.INCLUDE_CGROUP_RATIONALE = True
rag_core.INCLUDE_PER_VAR_RATIONALE = True

rag_core.TARGET = target          # reuse the target picked above, or set your own dict
out_obj = rag_core.run_single()   # prints rationales, saves JSON + XLSX

## Batch mode

Same as running `python3 rag_core.py` in a terminal: batch inference on the test list,
then automatic evaluation. Output filenames carry date/model/topk/fp/desc tags.

In [ ]:
# ── Batch mode: inference + evaluation ──
# Which dataset? -> set in the Setup cell at the top (IAM_LLM_BATCH_SHEET_PATH / IAM_LLM_GT_PATH).
# How many rows? -> N_LIMIT below.
N_LIMIT = 5     # smoke test with 5 targets first; set 0 to run the whole test list

rag_core.INCLUDE_CGROUP_RATIONALE = False   # rationales off in batch (cheaper, same as terminal)
rag_core.INCLUDE_PER_VAR_RATIONALE = False

dfv = rag_core.load_validation_xlsx(rag_core.BATCH_SHEET_PATH)
if N_LIMIT:
    dfv = dfv.head(N_LIMIT)

out_df = await rag_core.run_batch(dfv)   # top-level await works in Jupyter
out_df.to_csv(rag_core.OUT_BATCH_CSV, index=False)
print("predictions saved:", rag_core.OUT_BATCH_CSV)

rag_core.run_evaluation()
print("report saved:", rag_core.OUT_XLSX)